# Groove2Groove-PyTorch — Colab training

End-to-end training notebook for the hackathon. Workflow:

1. Mount Drive (for `dataset.zip` and for persisting checkpoints).
2. Clone the GitHub repo into `/content/`.
3. Copy `dataset.zip` from Drive to local SSD and unzip it (training off Drive is too slow).
4. Install Python deps.
5. Smoke-test (50 steps, tiny model) to confirm the pipeline.
6. Full training, with checkpoints written straight to Drive.
7. Generate a submission CSV from the best checkpoint and copy it to Drive.

**Before running**: in *Runtime → Change runtime type*, pick a GPU.

## 0. Edit these paths

In [ ]:
# === EDIT ME ============================================================
GITHUB_REPO   = "https://github.com/Nese2002/GenAI_Hackaton.git"  # public or use a token
GITHUB_BRANCH = "main"

# Path to dataset.zip on Drive (after mounting, Drive root is /content/drive/MyDrive)
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/GenAIHackaton/dataset.zip"

# Where on Drive to keep checkpoints + submissions (will be created if missing)
DRIVE_RUNS_DIR    = "/content/drive/MyDrive/GenAIHackaton/runs"
RUN_NAME          = "exp1"

# Training hyperparams (override anything you want from g2g_pytorch.config.Config)
BATCH_SIZE   = 16
LR           = 1e-3
MAX_STEPS    = 50_000
NUM_WORKERS  = 2
# ========================================================================

# Derived (don't edit unless you know why)
REPO_DIR    = "/content/hackaton"               # local clone path; change if you renamed the repo
DATASET_DIR = "/content/dataset"
LOGDIR      = f"{DRIVE_RUNS_DIR}/{RUN_NAME}"
print("REPO_DIR    =", REPO_DIR)
print("DATASET_DIR =", DATASET_DIR)
print("LOGDIR      =", LOGDIR)

## 1. Sanity-check the runtime

In [ ]:
!nvidia-smi || echo 'No GPU — switch runtime type to GPU before continuing.'
import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.exists(DRIVE_DATASET_ZIP), f"Can't find {DRIVE_DATASET_ZIP} — fix DRIVE_DATASET_ZIP above."
os.makedirs(LOGDIR, exist_ok=True)
print("dataset.zip size:", round(os.path.getsize(DRIVE_DATASET_ZIP) / 1e6, 1), "MB")

## 3. Clone the repo

If the repo is private, paste a personal access token into the URL: `https://<token>@github.com/<user>/<repo>.git`. Don't commit the token-bearing URL anywhere.

In [ ]:
import os, subprocess
if os.path.isdir(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    !cd $REPO_DIR && git fetch --depth=1 origin $GITHUB_BRANCH && git reset --hard origin/$GITHUB_BRANCH
else:
    !git clone --depth=1 -b $GITHUB_BRANCH $GITHUB_REPO $REPO_DIR
!ls $REPO_DIR
# In this repo the g2g_pytorch package sits at the top level (sibling to utility/).
CODE_DIR = REPO_DIR
assert os.path.isdir(f"{CODE_DIR}/g2g_pytorch"), \
    f"Couldn't find g2g_pytorch package under {CODE_DIR}. Adjust REPO_DIR."
assert os.path.isdir(f"{CODE_DIR}/utility"), \
    f"Couldn't find utility/ under {CODE_DIR}."
print("CODE_DIR =", CODE_DIR)

## 4. Copy + unzip the dataset to local SSD

Reading thousands of small NPZs straight off Drive is brutally slow. Copy the single zip locally, unzip once.

In [ ]:
import os, shutil, time
os.makedirs(DATASET_DIR, exist_ok=True)
if not os.path.exists(f"{DATASET_DIR}/manifest.csv"):
    t0 = time.time()
    print("Copying dataset.zip to local SSD...")
    shutil.copy(DRIVE_DATASET_ZIP, "/content/dataset.zip")
    print(f"  copy done in {time.time()-t0:.1f}s")
    print("Unzipping...")
    t0 = time.time()
    !unzip -q -o /content/dataset.zip -d /content/
    print(f"  unzip done in {time.time()-t0:.1f}s")
else:
    print("Dataset already present — skipping copy/unzip.")

# Find where manifest.csv landed. The zip might unpack directly into /content/ or into
# /content/<something>/. We walk /content/ MANUALLY, pruning Drive + Colab's own dirs --
# a recursive glob or `find` would otherwise descend into /content/drive/ (a FUSE mount
# over Google Drive) and hang for many minutes.
SKIP_DIRS = {"drive", "sample_data", "runs", ".config", ".ipynb_checkpoints", "hackaton"}
candidates = []
for root, subdirs, files in os.walk("/content"):
    subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
    if "manifest.csv" in files:
        candidates.append(os.path.join(root, "manifest.csv"))
assert candidates, "manifest.csv not found under /content/ after unzip."
DATASET_DIR = os.path.dirname(candidates[0])
print("DATASET_DIR =", DATASET_DIR)

# Cheap, bounded sanity prints (do NOT use shell find -- it would walk into /content/drive).
rolls_dir = os.path.join(DATASET_DIR, "rolls")
profiles_dir = os.path.join(DATASET_DIR, "style_profiles")
n_styles = len(os.listdir(rolls_dir)) if os.path.isdir(rolls_dir) else 0
n_profiles = len(os.listdir(profiles_dir)) if os.path.isdir(profiles_dir) else 0
print(f"style folders under rolls/: {n_styles}")
print(f"style profiles            : {n_profiles}")

## 5. Install Python deps

In [ ]:
# torch + numpy + pandas are already on Colab; we only need to make sure pandas/numpy are
# binary-compatible (we hit a numpy 2.x mismatch locally). On Colab this is almost always fine.
!pip install -q --upgrade pandas pretty_midi
import numpy, pandas, torch, pretty_midi
print("numpy", numpy.__version__, "pandas", pandas.__version__,
      "torch", torch.__version__, "pretty_midi", pretty_midi.__version__)

## 6. Smoke test (≈1 min)

Tiny model, 50 steps. Confirms the dataset path, the GPU path, and the val scorer work before you commit to a long run.

In [ ]:
!cd $CODE_DIR && python -m g2g_pytorch.train \
    --debug \
    --dataset-root $DATASET_DIR \
    --logdir /content/runs/smoke \
    --num-workers 0

## 7. Full training

Checkpoints land directly in `LOGDIR` on Drive so they survive runtime restarts. If your session disconnects, just re-run cells 0–5 and then this one with `--resume`.

*Tip*: `tail -f` the log file in another cell with `!tail -f $LOGDIR/train.log` if you redirect output there.

In [ ]:
!cd $CODE_DIR && python -m g2g_pytorch.train \
    --dataset-root $DATASET_DIR \
    --logdir $LOGDIR \
    --batch-size $BATCH_SIZE \
    --lr $LR \
    --max-steps $MAX_STEPS \
    --num-workers $NUM_WORKERS

### Resuming after a disconnect

Uncomment the cell below to keep training from the last checkpoint in `LOGDIR`.

In [ ]:
# !cd $CODE_DIR && python -m g2g_pytorch.train \
#     --dataset-root $DATASET_DIR \
#     --logdir $LOGDIR \
#     --batch-size $BATCH_SIZE \
#     --lr $LR \
#     --max-steps $MAX_STEPS \
#     --num-workers $NUM_WORKERS \
#     --resume $LOGDIR/model.pt

## 8. Generate a submission CSV

Reads the test split from `manifest.csv`, runs the trained model, writes the Kaggle wire format (`ID,notes`), and copies the result to Drive.

In [ ]:
SUBMISSION_LOCAL = f"/content/submission_{RUN_NAME}.csv"
SUBMISSION_DRIVE = f"{LOGDIR}/submission.csv"

!cd $CODE_DIR && python -m g2g_pytorch.infer \
    --dataset-root $DATASET_DIR \
    --ckpt $LOGDIR/model.pt \
    --output $SUBMISSION_LOCAL \
    --batch-size $BATCH_SIZE

import shutil, os
shutil.copy(SUBMISSION_LOCAL, SUBMISSION_DRIVE)
print("Wrote", SUBMISSION_DRIVE,
      f"({round(os.path.getsize(SUBMISSION_DRIVE)/1e6, 2)} MB)")

## 9. Spot-check the submission

Quick sanity look at the CSV — every test item should appear, with at least one note record.

In [ ]:
import pandas as pd, csv
sub = pd.read_csv(SUBMISSION_LOCAL)
print("rows:", len(sub), "unique IDs:", sub['ID'].nunique())
print("avg notes/item:", sub['notes'].str.count(';').add(1).mean().round(1))
print(sub.head(3))

# Cross-check against the test split in manifest.csv
with open(f"{DATASET_DIR}/manifest.csv") as f:
    expected = {r['item_id'] for r in csv.DictReader(f) if r['split'] == 'test'}
missing = expected - set(sub['ID'])
extra   = set(sub['ID']) - expected
print(f"expected {len(expected)} test items; missing {len(missing)}; extra {len(extra)}")
assert not missing, f"Submission is missing {len(missing)} test items, e.g. {sorted(missing)[:5]}"